# FlashEats - Validation and KPI definition

This notebook turns the leadership claim `Late Delivery Rate is 56%` into an explicit validation contract, compares competing definitions, and records known data limitations.

In [ ]:
from pathlib import Path
import json
import sqlite3
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'data' / 'raw' / 'sqlite' / 'flasheats.db').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'
FILES = RAW / 'files'
with sqlite3.connect(RAW / 'sqlite' / 'flasheats.db') as con:
    orders = pd.read_sql('select * from orders', con)
    restaurants = pd.read_sql('select * from restaurants', con)
    drivers = pd.read_sql('select * from drivers', con)
outcomes = pd.read_csv(FILES / 'order_outcomes.csv')
tickets = pd.read_csv(FILES / 'support_tickets.csv')
restaurant_status = pd.read_csv(FILES / 'restaurant_status.csv')
metric_definition = json.loads((FILES / 'client_metric_definitions.json').read_text(encoding='utf-8'))

for column in ['created_at', 'promised_eta', 'pickup_at', 'actual_delivery_at']:
    orders[column] = pd.to_datetime(orders[column], errors='coerce')
orders_one_row = orders.drop_duplicates('order_id', keep=False).copy()
exact_or_first = orders.drop_duplicates('order_id', keep='first').copy()
print('raw rows:', len(orders), 'unique order ids:', orders.order_id.nunique())
print('conflicting/duplicate order ids:', sorted(set(orders.loc[orders.order_id.duplicated(False), 'order_id'])))

## 1. Validation contract

A warning is not silently converted into a pass. The report distinguishes technical validity from unresolved business ownership.

In [ ]:
validation_contract = pd.DataFrame([
    ['Business grain', 'one row per order_id', orders.order_id.is_unique, 'WARN if duplicates conflict; quarantine before publishing'],
    ['Delivered completion', 'delivered orders have actual_delivery_at', int(((orders.final_status.astype(str).str.lower() == 'delivered') & orders.actual_delivery_at.isna()).sum()) == 0, 'WARN; outcome file has 37 unknown delivered outcomes'],
    ['ETA availability', 'promised_eta is present for measurable orders', orders.promised_eta.notna().all(), 'WARN; raw orders have 790 missing values'],
    ['Chronology', 'created <= pickup <= delivery', ((orders.pickup_at >= orders.created_at) & (orders.actual_delivery_at.isna() | (orders.actual_delivery_at >= orders.pickup_at))).all(), 'WARN/FAIL depending on KPI use'],
    ['KPI ownership', 'late definition is approved', False, 'UNKNOWN until Operations/Data Team confirms'],
], columns=['check', 'expectation', 'passed', 'action'])
validation_contract['status'] = validation_contract['passed'].map({True: 'PASS', False: 'WARN'})
display(validation_contract[['check', 'status', 'expectation', 'action']])

## 2. Compare KPI definitions

The outcome source contains a one-row-per-order classification and is the provisional source for the leadership KPI. Raw order timestamps are retained as an independent reconciliation check.

In [ ]:
outcome_measurable = outcomes[outcomes['delay_min'].notna()].copy()
outcome_measurable['delay_min'] = pd.to_numeric(outcome_measurable['delay_min'], errors='coerce')
outcome_measurable['late_flag'] = pd.to_numeric(outcome_measurable['late_flag'], errors='coerce')

kpi_table = pd.DataFrame([
    ['Outcome source: any delay > 0', int(outcome_measurable.late_flag.eq(1).sum()), len(outcome_measurable), outcome_measurable.late_flag.mean() * 100, 'Provisional KPI; excludes cancelled and unknown outcomes'],
    ['Outcome source: delay > 10 minutes', int(outcome_measurable.delay_min.gt(10).sum()), len(outcome_measurable), outcome_measurable.delay_min.gt(10).mean() * 100, 'Secondary meaningful-lateness metric'],
], columns=['definition', 'late_count', 'denominator', 'rate_pct', 'meaning'])
kpi_table['rate_pct'] = kpi_table['rate_pct'].round(2)
display(kpi_table)

raw = exact_or_first.copy()
raw['delay_min'] = (raw.actual_delivery_at - raw.promised_eta).dt.total_seconds() / 60
raw_measurable = raw[raw.actual_delivery_at.notna() & raw.promised_eta.notna()].copy()
raw_rate = raw_measurable.delay_min.gt(0).mean() * 100
print({'raw_order_timestamp_rate_pct': round(raw_rate, 2), 'raw_measurable': len(raw_measurable), 'outcome_rate_pct': round(outcome_measurable.late_flag.mean() * 100, 2), 'outcome_measurable': len(outcome_measurable)})

## 3. Cross-source integrity and data-quality evidence

In [ ]:
def mapping_result(label, source_values, target_values):
    values = source_values.dropna().astype(str)
    matched = values.isin(set(pd.Series(target_values).dropna().astype(str)))
    return {'relationship': label, 'rows_checked': len(values), 'coverage_pct': round(matched.mean() * 100, 2), 'unmapped_rows': int((~matched).sum())}

mapping_report = pd.DataFrame([
    mapping_result('orders.restaurant_id -> restaurants', exact_or_first.restaurant_id, restaurants.restaurant_id),
    mapping_result('orders.driver_id -> drivers', exact_or_first.driver_id, drivers.driver_id),
    mapping_result('outcomes.order_id -> orders', outcomes.order_id, exact_or_first.order_id),
    mapping_result('restaurant_status.order_id -> orders', restaurant_status.order_id, exact_or_first.order_id),
    mapping_result('support_tickets.order_id -> orders', tickets.loc[tickets.order_id.notna() & tickets.order_id.astype(str).str.strip().ne(''), 'order_id'], exact_or_first.order_id),
])
display(mapping_report)

print('Observed final_status values:', sorted(orders.final_status.dropna().astype(str).unique()))
print('Observed restaurant status values:', sorted(restaurant_status.status.dropna().astype(str).unique()))
support_unlinked = tickets.order_id.isna() | tickets.order_id.astype(str).str.strip().eq('')
print('Unlinked support tickets:', int(support_unlinked.sum()))
print('Outcome buckets:', outcomes.outcome_bucket.value_counts(dropna=False).to_dict())

## Decision gate

The 56.39% outcome-based KPI can be shown as provisional evidence, but the raw-order reconciliation remains a warning until the KPI owner confirms which source and denominator are authoritative.